In [ ]:
!pip install -q transformers torch datasets accelerate tqdm bitsandbytes huggingface-hub

In [ ]:
from huggingface_hub import login
login()

Part 3 - Flag set to True

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
SYSTEM_PROMPT = "You are a helpful AI assistant. Be concise and friendly."

USE_HISTORY = True  

MAX_CONTEXT_TOKENS = 2048
MAX_NEW_TOKENS = 256
SLIDING_WINDOW_TURNS = 8

TEMPERATURE = 0.7
TOP_P = 0.9

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Using device:", device)
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

print("Model loaded.\n")


def format_messages(messages):
    return "\n".join(
        f"[{m['role'].upper()}]: {m['content']}"
        for m in messages
    ) + "\n[ASSISTANT]:"

def count_tokens(messages):
    return len(tokenizer.encode(format_messages(messages), add_special_tokens=False))

def sliding_window(history):
    system = [m for m in history if m["role"] == "system"]
    non_system = [m for m in history if m["role"] != "system"]
    return system + non_system[-SLIDING_WINDOW_TURNS:]

def truncate_extra_turns(text):
    markers = ["\n[USER]:", "\n[ASSISTANT]:", "[USER]:", "[ASSISTANT]:"]
    first = None
    for m in markers:
        pos = text.find(m)
        if pos != -1:
            if first is None or pos < first:
                first = pos
    if first is not None:
        return text[:first].strip()
    return text.strip()

def generate_response(messages):
    prompt = format_messages(messages)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return truncate_extra_turns(raw)


chat_history = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat started! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    if USE_HISTORY:
        chat_history.append({"role": "user", "content": user_input})
        active_history = sliding_window(chat_history)

    else:
        active_history = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ]

    tokens_used = count_tokens(active_history)
    print(f"[Context tokens: {tokens_used}/{MAX_CONTEXT_TOKENS}]")

    start = time.time()
    response = generate_response(active_history)
    latency = time.time() - start

    print("Assistant:", response)
    print(f"(Latency: {latency:.2f}s)\n")

    if USE_HISTORY:
        chat_history.append({"role": "assistant", "content": response})

Part 3 - Flag set to False

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
SYSTEM_PROMPT = "You are a helpful AI assistant. Be concise and friendly."

USE_HISTORY = False

MAX_CONTEXT_TOKENS = 2048
MAX_NEW_TOKENS = 256
SLIDING_WINDOW_TURNS = 8

TEMPERATURE = 0.7
TOP_P = 0.9


device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Using device:", device)
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

print("Model loaded.\n")

def format_messages(messages):
    return "\n".join(
        f"[{m['role'].upper()}]: {m['content']}"
        for m in messages
    ) + "\n[ASSISTANT]:"

def count_tokens(messages):
    return len(tokenizer.encode(format_messages(messages), add_special_tokens=False))

def sliding_window(history):
    system = [m for m in history if m["role"] == "system"]
    non_system = [m for m in history if m["role"] != "system"]
    return system + non_system[-SLIDING_WINDOW_TURNS:]

def truncate_extra_turns(text):
    markers = ["\n[USER]:", "\n[ASSISTANT]:", "[USER]:", "[ASSISTANT]:"]
    first = None
    for m in markers:
        pos = text.find(m)
        if pos != -1:
            if first is None or pos < first:
                first = pos
    if first is not None:
        return text[:first].strip()
    return text.strip()

def generate_response(messages):
    prompt = format_messages(messages)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return truncate_extra_turns(raw)


chat_history = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat started! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    if USE_HISTORY:
        chat_history.append({"role": "user", "content": user_input})

        active_history = sliding_window(chat_history)

    else:
        active_history = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ]

    tokens_used = count_tokens(active_history)
    print(f"[Context tokens: {tokens_used}/{MAX_CONTEXT_TOKENS}]")

    start = time.time()
    response = generate_response(active_history)
    latency = time.time() - start

    print("Assistant:", response)
    print(f"(Latency: {latency:.2f}s)\n")

    if USE_HISTORY:
        chat_history.append({"role": "assistant", "content": response})